In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import pairwise_distances
from tqdm import tqdm

# Load and prepare the data
df = pd.read_csv("ml-latest-small/ratings.csv")
df = df.drop(columns=["timestamp"])
ratings_matrix = df.pivot(index="userId", columns="movieId", values="rating").fillna(0)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
def cosine_sim(matrix):
    return cosine_similarity(matrix.fillna(0))

def adjusted_cosine_sim(matrix):
    # Center by item (column) mean instead of user mean if you want item-based adjusted cosine
    matrix_centered = matrix.sub(matrix.mean(axis=1), axis=0)
    return cosine_similarity(matrix_centered.fillna(0))

def pearson_sim(matrix):
    # Use correlation matrix, which computes pairwise Pearson correlations
    matrix = matrix.T  # Each row becomes a user
    corr_matrix = matrix.corr(method='pearson').fillna(0).values
    return corr_matrix

def jaccard_sim(matrix):
    binarized = matrix.fillna(0).astype(bool).astype(int).values  # Convert to NumPy array
    return 1 - pairwise_distances(binarized, metric="jaccard")

In [53]:
def predict_rating(user_id, item_id, sim_matrix, rating_matrix, k=10):
    if item_id not in rating_matrix.columns:
        return np.nan

    sims = sim_matrix[user_id - 1]
    ratings = rating_matrix[item_id]

    # Mask out users who haven't rated the item
    mask = ratings > 0
    sims = sims[mask]
    ratings = ratings[mask]

    if len(ratings) == 0:
        return np.nan

    # Top-k neighbors
    top_k = np.argsort(sims)[-k:]
    sim_scores = sims[top_k]
    rating_scores = ratings.iloc[top_k]

    if sim_scores.sum() == 0:
        return np.nan

    return np.dot(sim_scores, rating_scores) / np.sum(sim_scores)


In [54]:
def evaluate(rating_matrix, sim_matrix, k, sample_size=1000):
    true_ratings = []
    pred_ratings = []

    test_users = np.random.choice(rating_matrix.index, size=sample_size)
    
    for user_id in tqdm(test_users):
        user_ratings = rating_matrix.loc[user_id]
        rated_items = user_ratings[user_ratings > 0].index
        if len(rated_items) == 0:
            continue
        item_id = np.random.choice(rated_items)
        true_rating = rating_matrix.at[user_id, item_id]

        pred = predict_rating(user_id, item_id, sim_matrix, rating_matrix, k)
        if not np.isnan(pred):
            true_ratings.append(true_rating)
            pred_ratings.append(pred)

    mae = mean_absolute_error(true_ratings, pred_ratings)
    rmse = mean_squared_error(true_ratings, pred_ratings) ** 0.5
    return mae, rmse


In [55]:
similarity_functions = {
    "cosine": cosine_sim,
    "adjusted_cosine": adjusted_cosine_sim,
    "pearson": pearson_sim,
    "jaccard": jaccard_sim,
}

k_values = [5, 10, 20, 30]
results = []

for name, sim_func in similarity_functions.items():
    print(f"Calculating {name} similarity...")
    sim_matrix = sim_func(ratings_matrix)

    for k in k_values:
        print(f"Evaluating with k = {k}")
        mae, rmse = evaluate(ratings_matrix, sim_matrix, k)
        results.append((name, k, mae, rmse))

# Convert to DataFrame
results_df = pd.DataFrame(results, columns=["similarity", "k", "mae", "rmse"])
print(results_df)


Calculating cosine similarity...
Evaluating with k = 5


100%|██████████| 1000/1000 [00:00<00:00, 4351.35it/s]


Evaluating with k = 10


100%|██████████| 1000/1000 [00:00<00:00, 4477.43it/s]


Evaluating with k = 20


...

Evaluating with k = 20


100%|██████████| 1000/1000 [00:00<00:00, 3936.39it/s]


Evaluating with k = 30


100%|██████████| 1000/1000 [00:00<00:00, 4708.90it/s]

         similarity   k       mae      rmse
0            cosine   5  0.405148  0.526389
1            cosine  10  0.515634  0.675131
2            cosine  20  0.554734  0.734142
3            cosine  30  0.594941  0.787606
4   adjusted_cosine   5  0.401131  0.538887
5   adjusted_cosine  10  0.501713  0.657481
6   adjusted_cosine  20  0.543792  0.709759
7   adjusted_cosine  30  0.559093  0.727255
8           pearson   5  0.396314  0.517305
9           pearson  10  0.490939  0.656723
10          pearson  20  0.552115  0.739173
11          pearson  30  0.555301  0.721550
12          jaccard   5  0.299349  0.404236
13          jaccard  10  0.400505  0.535580
14          jaccard  20  0.441449  0.597851
15          jaccard  30  0.484505  0.655740
